# 06 · Post-processing: Calibration and Foundation-Model Ensemble

**Owner:** Elena · **Reads:** Chronos-2 and TimesFM-3 forecasts (all 7 windows) · **Feeds:** notebook 05 (scoring)

This is the "beyond zero-shot" step the professor asked for. The foundation models' raw forecasts have two known problems:
1. **They run low.** Total forecast units are only about 73% of actual sales.
2. **Slow sellers get a flat zero.** The median of a mostly-zero product is zero, which plans no stock.

This notebook fixes those with simple, explainable corrections, and combines the two foundation models.

| New model | What it does | Goal |
|---|---|---|
| `<model>_calmae` | For each of 9 groups (category × sales speed), pick which percentile to forecast (10th–90th) and a cutoff below which the forecast rounds to zero | Lower error (MAE) |
| `<model>_calbias` | For each group, scale the median so total forecast units match total actual units | Unbiased totals, for inventory |
| `ens_fm` | Weighted average of Chronos-2 and TimesFM-3 (weight chosen on past windows) | Lower error |
| `ens_fm_calmae` | The ensemble, then the `calmae` correction | Lower error |

**No peeking (rolling origin):** corrections for a window are learned only from the windows **before** it.
`val_1` learns from the 4 backtests; `val_2` adds `val_1`; `test` adds `val_2`. The test window's actual sales are never used to choose anything.

New forecast files are saved in the shared format, so notebook 05 scores them automatically. Re-run 05 afterwards.

**Runtime:** CPU is fine. A few minutes.
**LightGBM:** the hybrid model and a LightGBM ensemble get added here once `02_lightgbm` exists.

## 0 · Settings

In [ ]:
SAMPLE      = False       # True = use the 100-product smoke-test forecasts
BASE_MODELS = ["chronos2_zs", "timesfm3_cov"]   # the best run of each foundation model (covariates helped TimesFM-3 only)
ENSEMBLE    = ("chronos2_zs", "timesfm3_cov")    # the two models to average
QUANTILE_GRID = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
ZERO_CUTOFFS  = [0.0, 0.25, 0.5]                  # forecast below this -> 0
WEIGHT_GRID   = [i / 10 for i in range(11)]       # ensemble weight on the first model
OVERWRITE   = True        # corrections depend on the inputs, so rewrite outputs each run
DRIVE_DIR   = None        # None = find AML_Term_Project in My Drive automatically; or paste a path

## 1 · Connect and find the shared folder

In [ ]:
import os, glob, json
import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not running in Colab: set DRIVE_DIR in Settings to your local AML_Term_Project folder.")

if DRIVE_DIR is None:
    found = (glob.glob("/content/drive/MyDrive/AML_Term_Project")
             or glob.glob("/content/drive/MyDrive/*/AML_Term_Project"))
    assert found, "Can't find AML_Term_Project in My Drive. Add the shortcut, or set DRIVE_DIR."
    DRIVE_DIR = found[0]
DATA_DIR = f"{DRIVE_DIR}/data"
FC_DIR   = f"{DATA_DIR}/forecasts_sample" if SAMPLE else f"{DATA_DIR}/forecasts"
FIG_DIR  = f"{DATA_DIR}/figures"
os.makedirs(FIG_DIR, exist_ok=True)
print("Forecast folder:", FC_DIR)

## 2 · Windows (in time order) and actual sales

In [ ]:
with open(f"{DATA_DIR}/windows.json") as fh:
    wjson = json.load(fh)
H = wjson["horizon"]
val1_start = pd.Timestamp(wjson["val_1"]["start"])
ORDER = [f"bt_{k}" for k in range(4, 0, -1)] + ["val_1", "val_2", "test"]
WIN_START = {f"bt_{k}": val1_start - pd.Timedelta(days=H * k) for k in range(4, 0, -1)}
WIN_START.update({w: pd.Timestamp(wjson[w]["start"]) for w in ["val_1", "val_2", "test"]})
TARGETS = ["val_1", "val_2", "test"]          # windows we produce corrected forecasts for

sales = pd.read_parquet(f"{DATA_DIR}/{'ca_sample100' if SAMPLE else 'ca_sales'}.parquet",
                        columns=["unique_id", "ds", "y", "is_christmas"])
sales["unique_id"] = sales["unique_id"].astype(str)
sales["ds"] = pd.to_datetime(sales["ds"])
sales = sales[sales["ds"] >= WIN_START["bt_4"]]
static = pd.read_parquet(f"{DATA_DIR}/ca_static.parquet")[["unique_id", "cat_id", "sales_speed"]]
static["unique_id"] = static["unique_id"].astype(str)
static["segment"] = static["cat_id"].astype(str) + " / " + static["sales_speed"].astype(str)
print(static["segment"].value_counts().sort_index())

## 3 · Load the foundation-model forecasts
Each file holds the median (`forecast`) and the 10th–90th percentiles (`q0.1` … `q0.9`).

In [ ]:
QCOLS = [f"q{q}" for q in QUANTILE_GRID]

def load_model(m):
    parts = []
    for w in ORDER:
        p = f"{FC_DIR}/{m}_{w}.parquet"
        if not os.path.exists(p):
            return None, w
        f = pd.read_parquet(p)
        f["window"] = w
        parts.append(f[["unique_id", "ds", "window", "forecast"] + [c for c in QCOLS if c in f.columns]])
    f = pd.concat(parts, ignore_index=True)
    f["unique_id"] = f["unique_id"].astype(str)
    f["ds"] = pd.to_datetime(f["ds"])
    f = (f.merge(sales, on=["unique_id", "ds"], how="inner")
          .merge(static[["unique_id", "segment"]], on="unique_id", how="left"))
    return f, None

data = {}
for m in sorted(set(BASE_MODELS) | set(ENSEMBLE)):
    f, missing = load_model(m)
    if f is None:
        print(f"SKIP {m}: missing {m}_{missing}.parquet in {FC_DIR}")
        continue
    data[m] = f
    print(f"{m:14s} {len(f):>10,} rows, windows: {sorted(f['window'].unique(), key=ORDER.index)}")
assert data, "No foundation-model forecasts found."

## 4 · The corrections
Every correction is learned on a **fit set** of earlier windows and applied to one **target** window.
Christmas Day (all stores closed) is left out of every fit set.

In [ ]:
def fit_rows(f, target):
    earlier = ORDER[:ORDER.index(target)]
    return f[f["window"].isin(earlier) & (f["is_christmas"] == 0)]

def fit_calmae(fit):
    # For each segment, choose (percentile, zero cutoff) with the lowest MAE on the fit windows.
    best = {}
    for q in [q for q in QUANTILE_GRID if f"q{q}" in fit.columns]:
        base = fit[f"q{q}"].to_numpy()
        for cut in ZERO_CUTOFFS:
            pred = np.where(base < cut, 0.0, base)
            err = pd.Series(np.abs(pred - fit["y"].to_numpy()), index=fit.index).groupby(fit["segment"]).mean()
            for seg, e in err.items():
                if seg not in best or e < best[seg][0]:
                    best[seg] = (e, q, cut)
    return {seg: (q, cut) for seg, (e, q, cut) in best.items()}

def apply_calmae(f, params):
    out = f["forecast"].to_numpy(dtype=float).copy()   # default: the median
    for seg, idx in f.groupby("segment").groups.items():
        q, cut = params.get(seg, (0.5, 0.0))
        base = f.loc[idx, f"q{q}"].to_numpy()
        out[f.index.get_indexer(idx)] = np.where(base < cut, 0.0, base)
    return out

def fit_calbias(fit):
    # For each segment, scale = total actual units / total forecast units (median) on the fit windows.
    g = fit.groupby("segment")
    return (g["y"].sum() / g["forecast"].sum().replace(0, np.nan)).fillna(1.0).clip(0.5, 3.0).to_dict()

def apply_calbias(f, scale):
    return f["forecast"].to_numpy() * f["segment"].map(scale).fillna(1.0).to_numpy()

## 5 · Calibrate each foundation model

In [ ]:
outputs, chosen = [], []
for m in [m for m in BASE_MODELS if m in data]:
    f = data[m]
    for target in TARGETS:
        fit = fit_rows(f, target)
        tgt = f[f["window"] == target].copy()
        p_mae = fit_calmae(fit)
        s_bias = fit_calbias(fit)
        for suffix, values in [("calmae", apply_calmae(tgt, p_mae)), ("calbias", apply_calbias(tgt, s_bias))]:
            o = tgt[["unique_id", "ds", "window"]].copy()
            o["model"] = f"{m}_{suffix}"
            o["forecast"] = np.clip(values, 0, None).astype("float32")
            outputs.append(o)
        for seg in sorted(p_mae):
            chosen.append({"model": m, "target window": target, "segment": seg,
                           "percentile": p_mae[seg][0], "zero cutoff": p_mae[seg][1],
                           "bias scale": round(s_bias.get(seg, 1.0), 2)})
chosen = pd.DataFrame(chosen)
print("What calibration chose for the TEST window (learned from the 6 earlier windows):")
chosen[chosen["target window"] == "test"].drop(columns="target window")

**How to read it:** a percentile above 0.5 means the model ran low for that group, so a higher percentile fits better.
A bias scale of 1.35 means forecasts were 35% too low in total, so they get multiplied by 1.35.

## 6 · Ensemble of Chronos-2 and TimesFM-3
Forecast = *w* × first model + (1 − *w*) × second model, for the median and every percentile.
The weight *w* is chosen on the earlier windows, then the `calmae` correction is applied to the ensemble too.

In [ ]:
a, b = ENSEMBLE
ens_weights = []
if a in data and b in data:
    cols = ["forecast"] + [c for c in QCOLS if c in data[a].columns and c in data[b].columns]
    both = data[a].merge(data[b][["unique_id", "ds", "window"] + cols], on=["unique_id", "ds", "window"], suffixes=("_a", "_b"))
    for target in TARGETS:
        fit = fit_rows(both, target)
        maes = {w: np.mean(np.abs(w * fit["forecast_a"] + (1 - w) * fit["forecast_b"] - fit["y"])) for w in WEIGHT_GRID}
        w = min(maes, key=maes.get)
        ens_weights.append({"target window": target, f"weight on {a}": w, f"weight on {b}": round(1 - w, 1)})
        mix = both[["unique_id", "ds", "window", "y", "is_christmas", "segment"]].copy()
        for c in cols:
            mix[c] = w * both[f"{c}_a"] + (1 - w) * both[f"{c}_b"]
        tgt = mix[mix["window"] == target]
        o = tgt[["unique_id", "ds", "window"]].copy(); o["model"] = "ens_fm"; o["forecast"] = tgt["forecast"].astype("float32")
        outputs.append(o)
        p = fit_calmae(fit_rows(mix, target))
        o2 = tgt[["unique_id", "ds", "window"]].copy(); o2["model"] = "ens_fm_calmae"
        o2["forecast"] = np.clip(apply_calmae(tgt, p), 0, None).astype("float32")
        outputs.append(o2)
    display(pd.DataFrame(ens_weights))
else:
    print(f"Ensemble skipped: need both {a} and {b}.")

## 7 · Save the new forecasts (shared format)

In [ ]:
new = pd.concat(outputs, ignore_index=True)
for (m, w), g in new.groupby(["model", "window"]):
    path = f"{FC_DIR}/{m}_{w}.parquet"
    if os.path.exists(path) and not OVERWRITE:
        continue
    g[["unique_id", "ds", "model", "window", "forecast"]].to_parquet(path, index=False)
print(f"Saved {new.groupby(['model', 'window']).ngroups} files for models:", sorted(new["model"].unique()))

## 8 · Quick before/after check
Not the official scoring (re-run notebook 05 for that). Same products and days for every row.
- **MAE:** lower is better.
- **Total forecast ÷ actual:** 1.00 means forecasts add up to real demand; below 1 means they run low.

In [ ]:
raw = [data[m][["unique_id", "ds", "window", "forecast", "y"]].assign(model=m) for m in data]
allf = pd.concat(raw + [new.merge(sales[["unique_id", "ds", "y"]], on=["unique_id", "ds"])], ignore_index=True)
allf = allf[allf["window"].isin(["val_2", "test"])]
common = allf.groupby(["window", "unique_id", "ds"])["model"].transform("nunique") == allf.groupby("window")["model"].transform("nunique")
allf = allf[common]
summary = (allf.assign(ae=(allf["forecast"] - allf["y"]).abs())
           .groupby(["model", "window"])[["ae", "forecast", "y"]]
           .apply(lambda g: pd.Series({"MAE": g["ae"].mean(), "Total forecast ÷ actual": g["forecast"].sum() / g["y"].sum()}))
           .unstack("window"))
summary.columns = [f"{a} ({b})" for a, b in summary.columns]
summary.sort_values(summary.columns[1]).round(3)

## 9 · Before/after picture (test window)

In [ ]:
import matplotlib.pyplot as plt
show = [m for m in [BASE_MODELS[0], f"{BASE_MODELS[0]}_calmae", f"{BASE_MODELS[0]}_calbias"] if m in allf["model"].unique()]
t = allf[(allf["window"] == "test") & allf["model"].isin(show)]
pick = (static[static["unique_id"].isin(t["unique_id"])].groupby("sales_speed", observed=True)["unique_id"].first())
fig, axes = plt.subplots(len(pick), 1, figsize=(10, 2.6 * len(pick)), sharex=True)
colors = {show[0]: "#A9B1BE"}
if len(show) > 1: colors[show[1]] = "#BF5700"
if len(show) > 2: colors[show[2]] = "#1f4e79"
for ax, (speed, uid) in zip(np.atleast_1d(axes), pick.items()):
    u = t[t["unique_id"] == uid]
    act = u.drop_duplicates("ds").sort_values("ds")
    ax.plot(act["ds"], act["y"], color="black", lw=1, label="actual")
    for m in show:
        um = u[u["model"] == m].sort_values("ds")
        ax.plot(um["ds"], um["forecast"], lw=2, color=colors[m], label=m)
    ax.set_title(f"{uid}  ({speed} seller)", fontsize=10); ax.set_ylabel("units")
np.atleast_1d(axes)[0].legend(frameon=False, fontsize=8, ncol=4)
plt.tight_layout()
if not SAMPLE: plt.savefig(f"{FIG_DIR}/22_calibration_before_after.png", dpi=150, bbox_inches="tight")
plt.show()

## What to report back
1. **Re-run notebook 05** so the new models get official scores and safety-stock dollars.
2. Download this notebook with outputs (File → Download .ipynb) and send it to Max, or commit it to `notebooks/06_post_processing.ipynb`.

**Questions to answer in the blog:**
- Does `calmae` lower the error? For which groups did it move away from the median?
- Does `calbias` bring total forecast ÷ actual close to 1.00, and what does that do to safety stock and coverage in 05?
- Does the ensemble beat both models on their own? What weight did it choose?